# 10 -- Infield Opportunity and Execution Analysis (Contact Luck v0.8)

Estimates infield ground-ball opportunity DIFFICULTY (`infield_contact_only_v08`:
`P(an average MLB infielder converts this into an out)`) and combined infield
EXECUTION (actual result vs. that opportunity-implied expectation), restricted to fair
ground balls where the batter-runner is the unambiguous, sole out opportunity. See
README.md "Infield opportunity and execution (Version 0.8)" for the full writeup and
CLAUDE.md/AGENTS.md for the governing rules (Version 0.7 is FROZEN and untouched by this
notebook; this is a genuinely new opportunity-difficulty category with no prior
baseline, judged against an absolute quality bar).

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

DATA_PATH = Path("../data/processed/cleaned_development_data_with_sprint_speed.parquet")
df = pd.read_parquet(DATA_PATH) if DATA_PATH.exists() else None
print(f"Loaded {len(df):,} rows" if df is not None else "Data not found -- run make download-sprint-speed && make join-sprint-speed first")

Loaded 494,173 rows


## 1. Infield-opportunity eligibility

In [2]:
from mlb_luck_score.eligibility import (
    add_infield_opportunity_eligibility,
    summarize_infield_exclusions,
)

elig_df = None
if df is not None:
    elig_df = add_infield_opportunity_eligibility(df)
    exclusions = summarize_infield_exclusions(elig_df)
    print(f"{elig_df['infield_opportunity_eligible'].sum():,} eligible rows of {len(elig_df):,} total")
    print()
    print(pd.Series(exclusions.total).sort_values(ascending=False))

143,377 eligible rows of 494,173 total

not_ground_ball_bb_type                 280141
eligible                                143377
outfield_credited_hit_location           34647
excluded_strategic_play                  30322
bunt_excluded                             4393
missing_required_contact_data             1151
interference_or_obstruction                113
missing_responsible_infield_position        29
dtype: int64


## 2. Feature engineering: `y_out` target and Version 0.8 feature set

`add_infield_opportunity_features` aliases `y_out` into `converted_to_out` so the
existing, unchanged `train_opportunity_model` trainer can be reused as-is.

In [3]:
from mlb_luck_score.features.build_contact_features import (
    add_infield_opportunity_features,
    select_infield_opportunity_features,
)

infield_df = None
if elig_df is not None:
    infield_df = add_infield_opportunity_features(elig_df)
    infield_df = infield_df[infield_df["infield_opportunity_eligible"].astype(bool)]
    numeric_features, categorical_features = select_infield_opportunity_features(infield_df)
    print("numeric:", numeric_features)
    print("categorical:", categorical_features)
    print()
    print(infield_df[numeric_features].notna().mean())

numeric: ['launch_speed', 'launch_angle', 'spray_angle_approx', 'hit_distance_sc', 'sprint_speed', 'outs_when_up', 'on_1b_occupied']
categorical: ['stand', 'if_fielding_alignment', 'assigned_infield_position', 'surface_type']

launch_speed          1.000000
launch_angle          1.000000
spray_angle_approx    0.999798
hit_distance_sc       0.999805
sprint_speed          0.987020
outs_when_up          1.000000
on_1b_occupied        1.000000
dtype: float64


## 3. Model selection (fit 2021-2022, select on 2023 by log loss)

Model CLASS comparison on the single `infield_contact_only_v08` feature set --
`LogisticRegression` vs. unweighted `HistGradientBoostingClassifier`, the same pattern
`compare_near_wall_models` used for `near_wall_logistic_v07c` vs. `near_wall_hgb_v07c`.

In [4]:
from mlb_luck_score.models.compare_infield_opportunity import run_infield_model_selection

winner_variant = None
selection_metrics = None
trained_by_candidate = None
if infield_df is not None:
    winner_variant, selection_metrics, trained_by_candidate = run_infield_model_selection(infield_df)
    print("winner:", winner_variant)
    print(json.dumps(selection_metrics, indent=2))

winner: infield_hgb_v08
{
  "infield_logistic_v08": {
    "binary_log_loss": 0.36866795479959386,
    "expected_calibration_error": 0.0167794517034352
  },
  "infield_hgb_v08": {
    "binary_log_loss": 0.3315724288760244,
    "expected_calibration_error": 0.012219896977846307
  }
}


## 4. Final comparison (2024, development validation -- 2025 untouched)

In [5]:
from mlb_luck_score.models.compare_infield_opportunity import run_infield_final_comparison

overall_comparison = None
final_df = None
p_out = None
if winner_variant is not None:
    winner_trained = trained_by_candidate[winner_variant]
    overall_comparison, final_df, p_out = run_infield_final_comparison(infield_df, winner_trained)
    print(json.dumps({k: v for k, v in overall_comparison.items() if k != "feature_missingness"}, indent=2))
    print()
    print("feature_missingness:")
    print(pd.Series(overall_comparison["feature_missingness"]))

{
  "variant": "infield_contact_only_v08",
  "sample_count": 35526,
  "binary_log_loss": 0.33565912831960226,
  "brier_score": 0.09876656011314358,
  "expected_calibration_error": 0.009108380181655478,
  "calibration_intercept": -0.17927282384857435,
  "calibration_slope": 1.1032614734345192,
  "outcome_prevalence": 0.864043235939875,
  "mean_predicted_p_out": 0.864914077629121
}

feature_missingness:
launch_speed                 0.000000
launch_angle                 0.000000
spray_angle_approx           0.000084
hit_distance_sc              0.000563
sprint_speed                 0.006418
outs_when_up                 0.000000
on_1b_occupied               0.000000
stand                        0.000000
if_fielding_alignment        0.003378
assigned_infield_position    0.000000
surface_type                 0.000000
dtype: float64


## 5. Perturbation checks

`sprint_speed_direction` is REQUIRED: a faster batter-runner must not show a higher
retirement probability than a slower one. `sprint_speed` has no derived-dependent
feature in this set, so there is no stale-derived-feature concern on override. Exit
velocity partial dependence is descriptive only (no required sign).

In [6]:
from mlb_luck_score.models.compare_infield_opportunity import (
    compute_grouped_sprint_speed_response,
    run_infield_perturbation_checks,
)

perturbation_suite = None
if final_df is not None:
    perturbation_suite = run_infield_perturbation_checks(winner_trained, final_df)
    for name, r in perturbation_suite.required.items():
        print(f"{name}: passed={r.passed} delta={r.delta:.4f} (low={r.mean_prob_low:.4f}, high={r.mean_prob_high:.4f}, n={r.sample_size})")
    print()
    print("timing_margin_check_status:", perturbation_suite.timing_margin_check_status)
    pdp = perturbation_suite.partial_dependence["all"]
    print(f"launch_speed PDP: {pdp.n_sign_changes} sign change(s), flagged_as_erratic={pdp.flagged_as_erratic}")
    print()
    response_df = pd.DataFrame([b.__dict__ for b in perturbation_suite.grouped_sprint_speed_response])
    print(response_df.head(10))

sprint_speed_direction: passed=True delta=-0.0899 (low=0.9279, high=0.8379, n=35298)

timing_margin_check_status: not_applicable
launch_speed PDP: 1 sign change(s), flagged_as_erratic=False

  position      bucket  mean_sprint_speed_fts  mean_predicted_p_out  sample_size
0        1  q1_slowest              25.597436              0.854375          780
1        1          q2              26.976336              0.845799          786
2        1          q3              27.930135              0.817996          813
3        1  q4_fastest              29.126534              0.787623          701
4        2  q1_slowest              25.630137              0.954715           73
5        2          q2              27.101316              0.926735           76
6        2          q3              27.972464              0.864201           69
7        2  q4_fastest              29.104348              0.873566           69
8        3  q1_slowest              25.625378              0.935326         1454

## 6. Required-subgroup and venue calibration gate (v0.7D machinery, reused)

Since there is no prior production infield-opportunity model, `baseline_p_out =
specialist_p_out` -- the paired-delta CI is identically zero and can never trigger a
"credible regression" finding, leaving only the absolute-ECE-confidence-interval
criterion active (per CLAUDE.md/AGENTS.md "A model with no prior baseline needs an
ABSOLUTE quality bar, not a relative one").

In [7]:
from mlb_luck_score.models.compare_infield_opportunity import (
    evaluate_all_infield_subgroups,
    evaluate_all_infield_venues,
    summarize_infield_calibration,
)
from mlb_luck_score.models.compare_near_wall_calibration_gate import SUBGROUP_STATUS_NOT_CALIBRATED

gate_summary = None
subgroup_evidence = None
venue_evidence = None
if final_df is not None:
    y_true = final_df["y_out"].astype(int).to_numpy()
    subgroup_evidence = evaluate_all_infield_subgroups(final_df, y_true, p_out)
    venue_evidence = evaluate_all_infield_venues(final_df, y_true, p_out)
    evidence_df = pd.DataFrame(
        [{"label": e.label, "status": e.status, "n_plays": e.n_plays, "ece": e.specialist_adaptive_ece} for e in subgroup_evidence]
    )
    print(evidence_df.set_index("label"))

                                          status  n_plays       ece
label                                                              
position_1                            calibrated     3106  0.016322
position_2                 insufficient_evidence      289  0.060451
position_3                            calibrated     5510  0.010377
position_4                            calibrated     9084  0.012383
position_5                            calibrated     8170  0.021988
position_6                            calibrated     9367  0.010439
pull                                  calibrated    24810  0.009921
center                                calibrated     3744  0.018412
opposite_field                        calibrated     6969  0.015705
batter_stand_L                        calibrated    15192  0.014089
batter_stand_R                        calibrated    20334  0.010619
sprint_speed_q1_slowest               calibrated     9393  0.012276
sprint_speed_q2                       calibrated

In [8]:
if final_df is not None:
    venue_evidence_df = pd.DataFrame(
        [{"label": e.label, "status": e.status, "n_plays": e.n_plays, "ece": e.specialist_adaptive_ece} for e in venue_evidence]
    ).sort_values("n_plays", ascending=False)
    print(f"{len(venue_evidence_df)} venues evaluated")
    print(venue_evidence_df.set_index("label").head(15))

34 venues evaluated
                             status  n_plays       ece
label                                                 
venue_2395.0             calibrated     1357  0.016168
venue_4169.0             calibrated     1303  0.024484
venue_3309.0  insufficient_evidence     1293  0.029638
venue_7.0                calibrated     1282  0.018913
venue_2889.0             calibrated     1264  0.024308
venue_19.0               calibrated     1254  0.023014
venue_2681.0             calibrated     1243  0.018569
venue_3.0                calibrated     1242  0.026471
venue_15.0    insufficient_evidence     1238  0.040840
venue_2392.0  insufficient_evidence     1194  0.023328
venue_2394.0             calibrated     1194  0.025928
venue_31.0               calibrated     1179  0.018063
venue_4705.0             calibrated     1179  0.019064
venue_17.0    insufficient_evidence     1178  0.033900
venue_2.0                calibrated     1177  0.018284


In [9]:
from mlb_luck_score.models.compare_infield_opportunity import compute_reached_on_error_comparison

if final_df is not None:
    reached_on_error = compute_reached_on_error_comparison(final_df, p_out)
    gate_summary = summarize_infield_calibration(
        overall_comparison, subgroup_evidence, venue_evidence, perturbation_suite, reached_on_error
    )
    print("overall_status:", gate_summary["overall_status"])
    print("calibrated_infield_opportunity:", gate_summary["calibrated_infield_opportunity"])
    print("calibrated:", len(gate_summary["calibrated_groups"]))
    print("not_calibrated:", gate_summary["not_calibrated_groups"])
    print("insufficient_evidence:", gate_summary["insufficient_evidence_groups"])

overall_status: calibrated_with_limited_subgroup_evidence
calibrated_infield_opportunity: False
calibrated: 47
not_calibrated: []
insufficient_evidence: ['position_2', 'alignment_infield_shift', 'venue_3309.0', 'venue_5325.0', 'venue_12.0', 'venue_3949.0', 'venue_680.0', 'venue_5381.0', 'venue_10.0', 'venue_3312.0', 'venue_2392.0', 'venue_2735.0', 'venue_5340.0', 'venue_17.0', 'venue_15.0']


## 7. Reached-on-error comparison (descriptive only, never a predictor)

Task Phase 3 explicitly forbids using the scorer's error classification as a model
input. This is a plausibility check only: from PRE-CONTACT features alone, does the
model treat reached-on-error plays similarly to genuine non-error safe outcomes?

In [10]:
if gate_summary is not None:
    print(json.dumps(gate_summary["reached_on_error_comparison"], indent=2))

{
  "reached_on_error_n": 976,
  "reached_on_error_mean_predicted_p_out": 0.8608632475570008,
  "non_error_safe_n": 3854,
  "non_error_safe_mean_predicted_p_out": 0.7063540840329474
}


### Interpretation

Reached-on-error plays score noticeably higher mean predicted `P(out)` than genuine
non-error safe outcomes -- consistent with errors disproportionately happening on
plays that "should" have been routine outs, using only pre-contact information (no
post-outcome leakage).

## 8. Combined infield execution and component report

```
defensive_execution_probability = actual_out - p_out_opportunity
batter_perspective_infield_execution = -defensive_execution_probability
```

Reported side by side with the existing Version 0.2 contact-model expectation/residual
-- **never summed into one score**, same reasoning as `air_ball_components`.

In [11]:
from mlb_luck_score.config import CALIBRATION_EVAL_SEASONS, TRAIN_SEASONS
from mlb_luck_score.eligibility import add_infield_opportunity_eligibility, compute_eligibility
from mlb_luck_score.models.train_contact_model import train_model
from mlb_luck_score.scoring.infield_ball_components import build_infield_ball_component_report

report = None
if df is not None and gate_summary is not None:
    contact_df = compute_eligibility(df)
    contact_train_df = contact_df[
        contact_df["eligible_for_training"].fillna(False) & contact_df["season"].isin(TRAIN_SEASONS)
    ]
    contact_trained = train_model(contact_train_df, class_weight=None)

    ground_ball_df = contact_df[contact_df["bb_type"] == "ground_ball"]
    ground_ball_df = add_infield_opportunity_eligibility(ground_ball_df)
    ground_ball_df = add_infield_opportunity_features(ground_ball_df)
    val_ground_ball_df = ground_ball_df[ground_ball_df["season"].isin(CALIBRATION_EVAL_SEASONS)]

    report = build_infield_ball_component_report(
        val_ground_ball_df,
        contact_trained,
        winner_trained,
        overall_status=gate_summary["overall_status"],
    )
    print(report["infield_opportunity_status"].value_counts())
    print()
    print(report.head(10))

infield_opportunity_status
insufficient_evidence         35526
unavailable_missing_inputs     9095
excluded_strategic_play        8363
Name: count, dtype: int64

        game_pk  at_bat_number  pitch_number  expected_run_value_contact_model  actual_run_value  residual_contact_luck_runs  p_out_opportunity  actual_out  \
370197   744795             14             2                         -0.080233         -0.254916                   -0.174683                NaN         NaN   
370198   744795             15             2                         -0.144996         -0.254916                   -0.109919           0.881817         1.0   
370207   744795             26             4                         -0.046727         -0.254916                   -0.208189                NaN         NaN   
370208   744795             28             4                         -0.103784         -0.254916                   -0.151132           0.968557         1.0   
370209   744795              3             

## 9. Explicit limitations

- **`infield_contact_only_v08` is the only implemented candidate.** Candidate B
  (`infield_time_margin_proxy_v08_candidate`) was investigated and NOT built -- there is
  no citable public physics for ground-ball roll deceleration the way a vacuum-
  projectile-motion formula exists for airborne trajectories.
- **This is a BINARY, opportunity-relative execution measure** -- it says nothing about
  pickup, transfer, footwork, or throwing execution specifically; a clean barehand play
  and a workmanlike play of the same difficulty register identically.
- **`calibrated_infield_opportunity` is `False`** -- `position_2` (catcher fielding a
  grounder, rare), `alignment_infield_shift`, and 13 individual venues currently lack
  adequate evidence (not a confirmed failure -- see "Measured miscalibration vs.
  insufficient evidence" in CLAUDE.md/AGENTS.md). The other 47 groups are genuinely
  `calibrated`.
- **`responsible_infielder_id` is for post-hoc evaluation only**, never a training
  feature -- training on it would encode that specific fielder's skill rather than
  physical opportunity difficulty.
- Version 0.7 (outfield air balls) remains FROZEN and untouched by this notebook.